In [2]:
"""
March Mania 2026 - LightGBM Prediction Model
=============================================
- Trains on regular season + tournament data from 2013–2023
- Predicts 2024 tournament matchup outcomes (held-out test set)
- No data leakage: features are built from each season's regular season only
- Seeds included as features
- Final 'FeaturesUsed' column lists every feature used
"""

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import log_loss, roc_auc_score

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
m_regular_season_detailed = pd.read_csv('/Users/shaurya/Downloads/MRegularSeasonDetailedResults.csv')
m_seeds                   = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneySeeds.csv')
m_tourney_detailed_result = pd.read_csv('/Users/shaurya/Downloads/MNCAATourneyDetailedResults.csv')

print("Regular Season shape :", m_regular_season_detailed.shape)
print("Seeds shape          :", m_seeds.shape)
print("Tourney Results shape:", m_tourney_detailed_result.shape)

# ─────────────────────────────────────────────
# 2. SEED PARSING  (e.g. "W01" → 1)
# ─────────────────────────────────────────────
def parse_seed(s):
    return int(''.join(filter(str.isdigit, s)))

m_seeds['SeedNum'] = m_seeds['Seed'].apply(parse_seed)

# ─────────────────────────────────────────────
# 3. BUILD TEAM SEASON STATS FROM REGULAR SEASON
#    Only data from the SAME season is used — no leakage.
# ─────────────────────────────────────────────
def compute_team_stats(reg_df, seasons):
    rows = reg_df[reg_df['Season'].isin(seasons)].copy()

    win = rows.rename(columns={
        'WTeamID':'TeamID','LTeamID':'OppID',
        'WScore':'Pts','LScore':'OppPts',
        'WFGM':'FGM','WFGA':'FGA','WFGM3':'FGM3','WFGA3':'FGA3',
        'WFTM':'FTM','WFTA':'FTA','WOR':'OR','WDR':'DR',
        'WAst':'Ast','WTO':'TO','WStl':'Stl','WBlk':'Blk','WPF':'PF',
    }).copy()
    win['Win'] = 1

    lose = rows.rename(columns={
        'LTeamID':'TeamID','WTeamID':'OppID',
        'LScore':'Pts','WScore':'OppPts',
        'LFGM':'FGM','LFGA':'FGA','LFGM3':'FGM3','LFGA3':'FGA3',
        'LFTM':'FTM','LFTA':'FTA','LOR':'OR','LDR':'DR',
        'LAst':'Ast','LTO':'TO','LStl':'Stl','LBlk':'Blk','LPF':'PF',
    }).copy()
    lose['Win'] = 0

    keep = ['Season','TeamID','Pts','OppPts','FGM','FGA','FGM3','FGA3',
            'FTM','FTA','OR','DR','Ast','TO','Stl','Blk','PF','Win']
    all_games = pd.concat([win[keep], lose[keep]], ignore_index=True)

    agg = all_games.groupby(['Season','TeamID']).agg(
        Games      =('Win','count'),
        WinRate    =('Win','mean'),
        AvgPts     =('Pts','mean'),
        AvgOppPts  =('OppPts','mean'),
        AvgFGM     =('FGM','mean'),
        AvgFGA     =('FGA','mean'),
        AvgFGM3    =('FGM3','mean'),
        AvgFGA3    =('FGA3','mean'),
        AvgFTM     =('FTM','mean'),
        AvgFTA     =('FTA','mean'),
        AvgOR      =('OR','mean'),
        AvgDR      =('DR','mean'),
        AvgAst     =('Ast','mean'),
        AvgTO      =('TO','mean'),
        AvgStl     =('Stl','mean'),
        AvgBlk     =('Blk','mean'),
        AvgPF      =('PF','mean'),
    ).reset_index()

    agg['FGPct']   = agg['AvgFGM']  / agg['AvgFGA'].replace(0, np.nan)
    agg['FG3Pct']  = agg['AvgFGM3'] / agg['AvgFGA3'].replace(0, np.nan)
    agg['FTPct']   = agg['AvgFTM']  / agg['AvgFTA'].replace(0, np.nan)
    agg['Margin']  = agg['AvgPts']  - agg['AvgOppPts']

    return agg.set_index(['Season','TeamID'])

# ─────────────────────────────────────────────
# 4. BUILD MATCHUP FEATURES
# ─────────────────────────────────────────────
def build_matchup_features(tourney_df, stats_df, seeds_df, seasons):
    records = []
    for _, row in tourney_df[tourney_df['Season'].isin(seasons)].iterrows():
        season = row['Season']
        w_id, l_id = row['WTeamID'], row['LTeamID']

        # Canonical ordering: lower TeamID = Team A
        if w_id < l_id:
            t_a, t_b, label = w_id, l_id, 1
        else:
            t_a, t_b, label = l_id, w_id, 0

        if (season, t_a) not in stats_df.index or (season, t_b) not in stats_df.index:
            continue

        sa = stats_df.loc[(season, t_a)]
        sb = stats_df.loc[(season, t_b)]

        seed_a_row = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==t_a)]
        seed_b_row = seeds_df[(seeds_df['Season']==season) & (seeds_df['TeamID']==t_b)]
        seed_a = seed_a_row['SeedNum'].values[0] if len(seed_a_row) else np.nan
        seed_b = seed_b_row['SeedNum'].values[0] if len(seed_b_row) else np.nan

        records.append({
            'Season': season, 'TeamA': t_a, 'TeamB': t_b, 'Label': label,

            # ── Seed features ──
            'SeedA':           seed_a,
            'SeedB':           seed_b,
            'SeedDiff':        seed_a - seed_b,   # negative = A better seeded

            # ── Difference features (A − B) ──
            'WinRateDiff':     sa['WinRate']  - sb['WinRate'],
            'PtsDiff':         sa['AvgPts']   - sb['AvgPts'],
            'OppPtsDiff':      sa['AvgOppPts']- sb['AvgOppPts'],
            'MarginDiff':      sa['Margin']   - sb['Margin'],
            'FGPctDiff':       sa['FGPct']    - sb['FGPct'],
            'FG3PctDiff':      sa['FG3Pct']   - sb['FG3Pct'],
            'FTPctDiff':       sa['FTPct']    - sb['FTPct'],
            'ORDiff':          sa['AvgOR']    - sb['AvgOR'],
            'DRDiff':          sa['AvgDR']    - sb['AvgDR'],
            'AstDiff':         sa['AvgAst']   - sb['AvgAst'],
            'TODiff':          sa['AvgTO']    - sb['AvgTO'],
            'StlDiff':         sa['AvgStl']   - sb['AvgStl'],
            'BlkDiff':         sa['AvgBlk']   - sb['AvgBlk'],
            'PFDiff':          sa['AvgPF']    - sb['AvgPF'],
            'GamesDiff':       sa['Games']    - sb['Games'],

            # ── Raw Team A stats ──
            'A_WinRate':       sa['WinRate'],
            'A_AvgPts':        sa['AvgPts'],
            'A_Margin':        sa['Margin'],
            'A_FGPct':         sa['FGPct'],
            'A_FG3Pct':        sa['FG3Pct'],
            'A_FTPct':         sa['FTPct'],
            'A_AvgOR':         sa['AvgOR'],
            'A_AvgDR':         sa['AvgDR'],
            'A_AvgAst':        sa['AvgAst'],
            'A_AvgTO':         sa['AvgTO'],
            'A_AvgStl':        sa['AvgStl'],
            'A_AvgBlk':        sa['AvgBlk'],

            # ── Raw Team B stats ──
            'B_WinRate':       sb['WinRate'],
            'B_AvgPts':        sb['AvgPts'],
            'B_Margin':        sb['Margin'],
            'B_FGPct':         sb['FGPct'],
            'B_FG3Pct':        sb['FG3Pct'],
            'B_FTPct':         sb['FTPct'],
            'B_AvgOR':         sb['AvgOR'],
            'B_AvgDR':         sb['AvgDR'],
            'B_AvgAst':        sb['AvgAst'],
            'B_AvgTO':         sb['AvgTO'],
            'B_AvgStl':        sb['AvgStl'],
            'B_AvgBlk':        sb['AvgBlk'],
        })
    return pd.DataFrame(records)

# ─────────────────────────────────────────────
# 5. TRAIN / TEST SPLIT  (no leakage)
#    Train: 2013–2023  |  Test: 2024
# ─────────────────────────────────────────────
TRAIN_SEASONS = list(range(2013, 2024))
TEST_SEASONS  = [2024]

all_seasons = TRAIN_SEASONS + TEST_SEASONS
stats = compute_team_stats(m_regular_season_detailed, all_seasons)

train_df = build_matchup_features(m_tourney_detailed_result, stats, m_seeds, TRAIN_SEASONS)
test_df  = build_matchup_features(m_tourney_detailed_result, stats, m_seeds, TEST_SEASONS)

print(f"\nTrain rows: {len(train_df)}  |  Test rows: {len(test_df)}")

# ─────────────────────────────────────────────
# 6. FEATURE COLUMNS
# ─────────────────────────────────────────────
FEATURE_COLS = [
    # Seeds
    'SeedA', 'SeedB', 'SeedDiff',
    # Difference features
    'WinRateDiff', 'PtsDiff', 'OppPtsDiff', 'MarginDiff',
    'FGPctDiff', 'FG3PctDiff', 'FTPctDiff',
    'ORDiff', 'DRDiff', 'AstDiff', 'TODiff', 'StlDiff', 'BlkDiff', 'PFDiff',
    'GamesDiff',
    # Raw Team A
    'A_WinRate', 'A_AvgPts', 'A_Margin', 'A_FGPct', 'A_FG3Pct', 'A_FTPct',
    'A_AvgOR', 'A_AvgDR', 'A_AvgAst', 'A_AvgTO', 'A_AvgStl', 'A_AvgBlk',
    # Raw Team B
    'B_WinRate', 'B_AvgPts', 'B_Margin', 'B_FGPct', 'B_FG3Pct', 'B_FTPct',
    'B_AvgOR', 'B_AvgDR', 'B_AvgAst', 'B_AvgTO', 'B_AvgStl', 'B_AvgBlk',
]

X_train = train_df[FEATURE_COLS]
y_train = train_df['Label']
X_test  = test_df[FEATURE_COLS]
y_test  = test_df['Label']

# ─────────────────────────────────────────────
# 7. TRAIN LIGHTGBM
# ─────────────────────────────────────────────
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_valid = lgb.Dataset(X_test,  label=y_test, reference=lgb_train)

params = {
    'objective':         'binary',
    'metric':            'binary_logloss',
    'boosting_type':     'gbdt',
    'num_leaves':        31,
    'learning_rate':     0.05,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'min_child_samples': 10,
    'lambda_l1':         0.1,
    'lambda_l2':         0.1,
    'verbose':           -1,
    'seed':              42,
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_valid],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(50)],
)

# ─────────────────────────────────────────────
# 8. EVALUATE
# ─────────────────────────────────────────────
y_pred_prob = model.predict(X_test)
y_pred_bin  = (y_pred_prob >= 0.5).astype(int)

ll  = log_loss(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)
acc = (y_pred_bin == y_test).mean()

print(f"\n{'='*45}")
print(f"  2024 Tournament Results")
print(f"{'='*45}")
print(f"  Log Loss : {ll:.4f}")
print(f"  ROC AUC  : {auc:.4f}")
print(f"  Accuracy : {acc:.4f}  ({int(acc*len(y_test))}/{len(y_test)} correct)")

# ─────────────────────────────────────────────
# 9. PREDICTIONS + FEATURE LIST COLUMN
# ─────────────────────────────────────────────
results_df = test_df[['Season','TeamA','TeamB','Label'] + FEATURE_COLS].copy()
results_df['PredProb_TeamA_Wins'] = y_pred_prob
results_df['PredWinner']   = results_df.apply(
    lambda r: r['TeamA'] if r['PredProb_TeamA_Wins'] >= 0.5 else r['TeamB'], axis=1)
results_df['ActualWinner'] = results_df.apply(
    lambda r: r['TeamA'] if r['Label'] == 1 else r['TeamB'], axis=1)
results_df['Correct']      = (results_df['PredWinner'] == results_df['ActualWinner']).astype(int)

# One column listing every feature used (same for all rows)
results_df['FeaturesUsed'] = ' | '.join(FEATURE_COLS)

# ─────────────────────────────────────────────
# 10. FEATURE IMPORTANCE
# ─────────────────────────────────────────────
importance = pd.DataFrame({
    'Feature':    model.feature_name(),
    'Importance': model.feature_importance(importance_type='gain'),
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\nTop 15 Features by Gain:")
print(importance.head(15).to_string(index=False))

# ─────────────────────────────────────────────
# 11. SAVE
# ─────────────────────────────────────────────
results_df.to_csv('/Users/shaurya/Downloads/march_mania_2024_predictions.csv', index=False)
importance.to_csv('/Users/shaurya/Downloads/march_mania_feature_importance.csv', index=False)

print(f"\n✅ Predictions saved  → /Users/shaurya/Downloads/march_mania_2024_predictions.csv")
print(f"✅ Feature importance → /Users/shaurya/Downloads/march_mania_feature_importance.csv")
print(f"\nTotal features used : {len(FEATURE_COLS)}")
print("Feature list        :", FEATURE_COLS)

Regular Season shape : (122775, 34)
Seeds shape          : (2626, 3)
Tourney Results shape: (1449, 34)

Train rows: 669  |  Test rows: 67
[50]	valid_0's binary_logloss: 0.565414
[100]	valid_0's binary_logloss: 0.561543
[150]	valid_0's binary_logloss: 0.540044
[200]	valid_0's binary_logloss: 0.547223

  2024 Tournament Results
  Log Loss : 0.5302
  ROC AUC  : 0.7984
  Accuracy : 0.7164  (48/67 correct)

Top 15 Features by Gain:
   Feature  Importance
  SeedDiff  977.874030
MarginDiff  398.836853
  A_AvgStl  327.481737
   StlDiff  284.440371
   BlkDiff  272.189966
    PFDiff  268.972774
   A_AvgTO  263.640444
   A_AvgDR  250.750218
  B_AvgPts  239.226683
   A_FGPct  236.787474
  A_AvgBlk  221.968049
OppPtsDiff  221.364157
    ORDiff  217.508422
   B_FGPct  214.644013
  B_Margin  206.348068

✅ Predictions saved  → /Users/shaurya/Downloads/march_mania_2024_predictions.csv
✅ Feature importance → /Users/shaurya/Downloads/march_mania_feature_importance.csv

Total features used : 42
Feature li